In [1]:
import torch
import einops

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [4]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        # first, ensure no cyclic paths can contribute
        dp[:, i, :i] = float('-inf')
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [5]:
def fix_probs(logprobs, mask):
    # assumes probs is already in log space
    # and is a square matrix
    # updates probs so that the sum of each row is 1
    # and any available probability mass is 
    # distributed evenly among the non-masked entries
    batch_size, l, _ = logprobs.shape

    # any part of the mask where the value is not 0, we mask out
    logprobs = logprobs.masked_fill(mask != 0, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining
    probnonzero = torch.sum(mask == 0, dim=-1)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)

    # at this point, it is mostly correct, however if there 
    # are rows where the number of non zeros (probnonzero)
    # is 0, then we get infinities at those positions, which
    # are obviously wrong, so we keep only positions
    # where we haven't masked out
    probsmatrix = probsmatrix.masked_fill(mask != 0, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [6]:
m = 7
l = 9
padded_l = l + 3
vocab_size = 9

# the 0s after the 8 are padding
target = [0, 1, 2, 3, 4, 5, 8, 0, 0, 0, 0, 0]
assert len(target) == padded_l
target = torch.tensor(target)

In [7]:
transition_matrix = torch.zeros((l, l))

In [8]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.1 #prob
    ),
    (
        (2, 1), #row, col
        0.8 #prob
    ),
    (
        (2, 9), #row, col
        0.1 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),
]

In [9]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [10]:
token_probs = torch.zeros((l, vocab_size))

In [11]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.8 #prob
    ),
    (
        (1, 3), #row, col
        0.1
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.1 #prob
    ),
    (
        (2, 4), #row, col
        0.1 #prob
    ),
    (
        (2, 6), #row, col
        0.1 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    )
]

In [12]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [13]:
padded_transition_matrix = torch.zeros((padded_l, padded_l))

In [14]:
padded_transition_matrix[:l, :l] = transition_matrix

In [15]:
padded_token_probs = torch.zeros((padded_l, vocab_size))

In [16]:
padded_token_probs[:l, :] = token_probs

In [17]:
# important, this is part of the mask process
# in real world, we can figure out how big the mask
# should be because each output should have a `transition_matrix`
# of shape batch_size x padded_l x padded_l
mask = torch.tril(torch.ones((padded_l, padded_l)))

In [18]:
# unsqueeze transition matrix, token probs, and targets to mimic batch size of 1
# padded_transition_matrix = padded_transition_matrix.unsqueeze(0)
# padded_token_probs = padded_token_probs.unsqueeze(0)
# target = target.unsqueeze(0)
batched_transition_matrix = torch.zeros((2, padded_l, padded_l))
batched_transition_matrix[0] = padded_transition_matrix
batched_transition_matrix[1] = padded_transition_matrix

batched_token_probs = torch.zeros((2, padded_l, vocab_size))
batched_token_probs[0] = padded_token_probs
batched_token_probs[1] = padded_token_probs

batched_target = torch.zeros((2, target.shape[-1])).type(target.dtype)
batched_target[0] = target
batched_target[1] = target

In [19]:
batched_transition_matrix[0]

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.8000, 0.0000, 0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [20]:
# this is another part of the mask process, though it is only used to select
# the appropriate value in the loss output
# this cannot be determined by the system, it needs to be given
# from data loader (in this case we hardcoded)
target_lens = torch.tensor([m, m])

In [21]:
# this is also a part of the masking process, used to help create mask
# for transition matrix. this *could* be determined by multiplying
# the target_lens by some factor (which is a hyperparameter), or
# it could also be given either from data loader or dynamically
# set via some function of target_lens that works outside of the model
vertex_lens = torch.tensor([l, l])

In [22]:
# an important part of the masking process, we can determine the needed
# padded_l from `transition_matrix`, and `vertex_lens` is some
# function of `target_lens` (or something that is given)
vertex_lens_mask = torch.arange(padded_l).repeat(len(vertex_lens), 1) < vertex_lens.unsqueeze(-1)

In [23]:
batched_transition_matrix = torch.log(batched_transition_matrix)
batched_token_probs = torch.log(batched_token_probs)

In [24]:
# we are given `batched_transition_matrix` as the model output,
# so we can use it to create an appropriate mask, first we start
# by creating a matrix of all ones that is the same shape as
# `batched_transition_matrix`
batched_transition_mask = torch.ones_like(batched_transition_matrix)

In [25]:
vertex_lens_mask.shape

torch.Size([2, 12])

In [26]:
# we then use `vertex_lens_mask` to select the appropriate
# values in `batched_transition_mask` and set them to 0,
# so those values are not masked
batched_transition_mask.transpose(1,2)[vertex_lens_mask] = 0

In [27]:
batched_transition_mask.shape # torch.Size([2, 12, 12]), int of 1s where we want to mask and 0s where we want to keep

torch.Size([2, 12, 12])

In [28]:
mask.shape # torch.Size([12, 12]), int of 1s where we want to mask and 0s where we want to keep

torch.Size([12, 12])

In [29]:
mask

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [30]:
# combine mask with batched_transition_mask,
# this is because `mask` prevents cycles in the dag,
# while the batched_transition_mask prevents 'real'
# vertices from transitioning to padding vertices
batched_transition_mask = batched_transition_mask + mask

In [31]:
# however, this means that may be some probability mass
# that are assigned from real vertices to padding vertices
# or edges that can cause cycles, so if we just mask out
# as is, we might not get a valid probability distribution
# for the transitions we care about, so we need to fix first
# batched_transition_matrix = fix_probs(batched_transition_matrix, batched_transition_mask)

In [32]:
dp = dag_loss(batched_target, batched_transition_matrix, batched_token_probs)

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [40]:
dp[0]

tensor([[ -0.2231,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,  -1.7838,  -2.1893,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [ -4.3095,     -inf,  -4.3095,  -4.4918,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,  -7.8161,     -inf,  -4.4149,  -6.7944,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [-10.3418,     -inf,     -inf,     -inf,  -4.5202,     -inf,  -9.7902,
             -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf, -13.8484,     -inf,     -inf,     -inf,  -5.7242,  -7.5160,
         -11.3996,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
          -9.1254,  -6.0774,     -inf,     -inf,     -inf],
        [    -inf,     -inf

In [33]:
torch.exp(dp[0])

tensor([[8.0000e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.6800e-01, 1.1200e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3440e-02, 0.0000e+00, 1.3440e-02, 1.1200e-02, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 4.0320e-04, 0.0000e+00, 1.2096e-02, 1.1200e-03, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [3.2256e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0886e-02, 0.0000e+00,
         5.6000e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.6768e-07, 0.0000e+00, 0.0000e+00, 0.0000e+00, 3.2659e-03,
         5.4432e-04, 1.1200e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0

In [34]:
dp_values = vector_gather(dp, target_lens - 1)

In [35]:
torch.exp(dp_values)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0001, 0.0023,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0001, 0.0023,
         0.0000, 0.0000, 0.0000]])

In [36]:
dp_values

tensor([[   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -9.1254,
         -6.0774,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -9.1254,
         -6.0774,    -inf,    -inf,    -inf]])

In [37]:
values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))

In [38]:
values

tensor([[-6.0774],
        [-6.0774]])

In [39]:
torch.exp(values)

tensor([[0.0023],
        [0.0023]])